In [1]:
# -*- coding: utf-8 -*-
"""
PLOS ONE 财务舞弊识别论文 —— 8 张图完整绘制脚本
环境: Python 3.8.8, matplotlib 3.3.4, seaborn 0.11.1
输出: D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.calibration import calibration_curve

# ==================== 路径 ====================
BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

# 新建独立图表文件夹（与实验结果分离）
FIG_DIR = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表"
os.makedirs(FIG_DIR, exist_ok=True)

# ==================== 全局样式 ====================
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = False

RNG = np.random.RandomState(42)


def save(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, bbox_inches='tight', dpi=300)
    fig.savefig(pdf, bbox_inches='tight')
    plt.close(fig)
    print("[saved]", png)


# ==================== 图 1: ROC 曲线 ====================
def fig1_roc():
    df = pd.read_csv(os.path.join(BASE, "对比_OOF_predictions.csv"))
    y = df['y_true'].values
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    configs = [
        ('Feature Set A (57 features)',
         ['Base_LightGBM', 'Base_XGBoost', 'Base_RandomForest']),
        ('Feature Set A+B+C (83 features)',
         ['Full_LightGBM', 'Full_XGBoost', 'Full_RandomForest']),
    ]
    colors = {'LightGBM': '#1f77b4', 'XGBoost': '#ff7f0e', 'RandomForest': '#2ca02c'}
    for ax, (title, cols) in zip(axes, configs):
        for col in cols:
            model = col.split('_', 1)[1]
            fpr, tpr, _ = roc_curve(y, df[col].values)
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, label=f'{model} (AUC={roc_auc:.4f})',
                    color=colors[model], lw=1.8)
        ax.plot([0, 1], [0, 1], 'k--', lw=1)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(title, fontsize=11)
        ax.legend(loc='lower right', fontsize=9, frameon=True)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    plt.tight_layout()
    save(fig, "fig1_roc")


# ==================== 图 2: PR 曲线 ====================
def fig2_pr():
    df = pd.read_csv(os.path.join(BASE, "对比_OOF_predictions.csv"))
    y = df['y_true'].values
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    configs = [
        ('Feature Set A (57 features)',
         ['Base_LightGBM', 'Base_XGBoost', 'Base_RandomForest']),
        ('Feature Set A+B+C (83 features)',
         ['Full_LightGBM', 'Full_XGBoost', 'Full_RandomForest']),
    ]
    colors = {'LightGBM': '#1f77b4', 'XGBoost': '#ff7f0e', 'RandomForest': '#2ca02c'}
    baseline = y.mean()
    for ax, (title, cols) in zip(axes, configs):
        for col in cols:
            model = col.split('_', 1)[1]
            prec, rec, _ = precision_recall_curve(y, df[col].values)
            ap = average_precision_score(y, df[col].values)
            ax.plot(rec, prec, label=f'{model} (PR-AUC={ap:.4f})',
                    color=colors[model], lw=1.8)
        ax.axhline(baseline, color='gray', ls='--', lw=1,
                   label=f'Baseline ({baseline:.3f})')
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.set_title(title, fontsize=11)
        ax.legend(loc='upper right', fontsize=9)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    plt.tight_layout()
    save(fig, "fig2_pr")


# ==================== 图 3: 校准曲线 ====================
def fig3_calibration():
    df = pd.read_csv(os.path.join(BASE, "Optuna_calibrated_predictions.csv"))
    y = df['y_true'].values
    raw = df['y_prob_raw'].values
    cal = df['y_prob_calibrated'].values

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    for probs, label, color in [(raw, 'Raw probability', '#d62728'),
                                (cal, 'Isotonic calibrated', '#1f77b4')]:
        pt, pp = calibration_curve(y, probs, n_bins=10, strategy='quantile')
        ax.plot(pp, pt, 'o-', label=label, color=color, lw=1.8, ms=5)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed fraud rate')
    ax.set_title('Calibration curve (Optuna LightGBM, OOF)', fontsize=11)
    ax.legend(loc='upper left', fontsize=9)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    plt.tight_layout()
    save(fig, "fig3_calibration")


# ==================== 图 4: SHAP 全局 bar ====================
def fig4_shap_bar():
    df = pd.read_csv(os.path.join(BASE, "SHAP_global_importance.csv"))
    top = df.nsmallest(20, 'Rank').iloc[::-1]
    colors = ['#d62728' if m else '#1f77b4' for m in top['IsMDA']]
    fig, ax = plt.subplots(figsize=(8, 7.5))
    ax.barh(top['Feature'], top['MeanAbsSHAP'], color=colors,
            edgecolor='black', linewidth=0.4)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('Top 20 features by global SHAP importance', fontsize=11)
    legend_elements = [
        Line2D([0], [0], color='#d62728', lw=6, label='MD&A text feature'),
        Line2D([0], [0], color='#1f77b4', lw=6, label='Financial / non-financial'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    ax.tick_params(axis='y', labelsize=9)
    plt.tight_layout()
    save(fig, "fig4_shap_bar")


# ==================== 图 5: SHAP beeswarm ====================
def _beeswarm_jitter(x, n_bins=80, rng=None):
    if rng is None:
        rng = np.random.RandomState(42)
    hist, edges = np.histogram(x, bins=n_bins)
    bin_idx = np.digitize(x, edges) - 1
    bin_idx = np.clip(bin_idx, 0, len(hist) - 1)
    density = hist[bin_idx].astype(float)
    density = density / (density.max() + 1e-12)
    jitter = (rng.rand(len(x)) - 0.5) * 0.9 * np.sqrt(density)
    return jitter


def fig5_shap_beeswarm():
    sv = pd.read_csv(os.path.join(BASE, "SHAP_values_all.csv"))
    fv = pd.read_csv(os.path.join(BASE, "SHAP_feature_values.csv"))
    gi = pd.read_csv(os.path.join(BASE, "SHAP_global_importance.csv"))

    assert np.array_equal(sv['y_true'].values, fv['y_true'].values), "y_true 不对齐"

    top15 = gi.nsmallest(15, 'Rank')['Feature'].tolist()

    n = len(sv)
    n_show = min(3000, n)
    idx = RNG.choice(n, n_show, replace=False)

    fig, ax = plt.subplots(figsize=(9, 8))
    cmap = plt.get_cmap('coolwarm')

    for i, feat in enumerate(top15):
        y_pos = len(top15) - 1 - i
        shap_vals = sv[feat].values[idx]
        feat_vals = fv[feat].values[idx]
        fmin, fmax = np.nanmin(feat_vals), np.nanmax(feat_vals)
        if fmax - fmin < 1e-12:
            c = np.zeros_like(feat_vals)
        else:
            c = (feat_vals - fmin) / (fmax - fmin)
        c = np.clip(c, 0, 1)
        jitter = _beeswarm_jitter(shap_vals, rng=np.random.RandomState(42 + i))
        ax.scatter(shap_vals, y_pos + jitter, c=c, cmap=cmap,
                   s=4, alpha=0.65, edgecolors='none', vmin=0, vmax=1)

    ax.set_yticks(range(len(top15)))
    ax.set_yticklabels(top15[::-1], fontsize=9)
    ax.axvline(0, color='gray', lw=0.8, ls='--')
    ax.set_xlabel('SHAP value (impact on fraud probability)')
    ax.set_title('SHAP beeswarm - Top 15 features', fontsize=11)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label('Feature value (normalized)', fontsize=9)
    cbar.set_ticks([0, 1]); cbar.set_ticklabels(['Low', 'High'])

    plt.tight_layout()
    save(fig, "fig5_shap_beeswarm")


# ==================== 图 6: SHAP dependence ====================
def fig6_shap_dependence():
    sv = pd.read_csv(os.path.join(BASE, "SHAP_values_all.csv"))
    fv = pd.read_csv(os.path.join(BASE, "SHAP_feature_values.csv"))

    feat = 'TextualSimilarity'
    shap_vals = sv[feat].values
    feat_vals = fv[feat].values

    mask = ~(np.isnan(shap_vals) | np.isnan(feat_vals))
    shap_vals = shap_vals[mask]
    feat_vals = feat_vals[mask]

    n = len(shap_vals)
    n_show = min(8000, n)
    idx = RNG.choice(n, n_show, replace=False)
    x = feat_vals[idx]
    y = shap_vals[idx]

    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    ax.scatter(x, y, s=4, alpha=0.25, color='#1f77b4', edgecolors='none')

    n_bins = 20
    bins = np.quantile(x, np.linspace(0, 1, n_bins + 1))
    bins = np.unique(bins)
    bin_centers, bin_means = [], []
    for j in range(len(bins) - 1):
        m = (x >= bins[j]) & (x < bins[j + 1])
        if m.sum() > 10:
            bin_centers.append(np.median(x[m]))
            bin_means.append(y[m].mean())
    ax.plot(bin_centers, bin_means, 'r-', lw=2, label='Binned mean SHAP')

    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_xlabel('TextualSimilarity (cosine similarity vs. previous year MD&A)')
    ax.set_ylabel('SHAP value for TextualSimilarity')
    ax.set_title('SHAP dependence plot: TextualSimilarity', fontsize=11)
    ax.legend(loc='best', fontsize=9)
    plt.tight_layout()
    save(fig, "fig6_shap_dependence")


# ==================== 图 7: SHAP 5 折稳定性 ====================
def fig7_shap_stability():
    df = pd.read_csv(os.path.join(BASE, "SHAP_stability_5fold.csv"))
    df = df.rename(columns={'Unnamed: 0': 'Feature'})
    top = df.nsmallest(15, 'rank_mean').iloc[::-1]

    fig, ax = plt.subplots(figsize=(8, 6.5))
    y_pos = np.arange(len(top))
    ax.errorbar(top['mean'], y_pos, xerr=top['std'],
                fmt='o', color='#1f77b4', ecolor='#7f7f7f',
                elinewidth=1.2, capsize=3, ms=5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top['Feature'], fontsize=9)
    ax.set_xlabel('Mean |SHAP value| across 5 folds')
    ax.set_title('SHAP importance stability across 5 folds (Top 15)', fontsize=11)
    ax.grid(axis='x', linestyle=':', alpha=0.5)
    plt.tight_layout()
    save(fig, "fig7_shap_stability")


# ==================== 图 8: SMOTE vs class_weight SHAP ====================
def fig8_smote_vs_cw():
    cw = pd.read_csv(os.path.join(BASE, "SHAP_global_importance.csv"))
    sm = pd.read_csv(os.path.join(BASE, "SMOTE_SHAP_global_importance.csv"))

    cw_top = cw.nsmallest(15, 'Rank').iloc[::-1]
    sm_top = sm.nsmallest(15, 'Rank').iloc[::-1]

    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))

    axes[0].barh(cw_top['Feature'], cw_top['MeanAbsSHAP'],
                 color=['#d62728' if m else '#1f77b4' for m in cw_top['IsMDA']],
                 edgecolor='black', linewidth=0.4)
    axes[0].set_title('class_weight = balanced (main model)', fontsize=11)
    axes[0].set_xlabel('Mean |SHAP value|')
    axes[0].tick_params(axis='y', labelsize=9)

    axes[1].barh(sm_top['Feature'], sm_top['MeanAbsSHAP'],
                 color=['#d62728' if m else '#1f77b4' for m in sm_top['IsMDA']],
                 edgecolor='black', linewidth=0.4)
    axes[1].set_title('SMOTE (1:1 oversampling)', fontsize=11)
    axes[1].set_xlabel('Mean |SHAP value|')
    axes[1].tick_params(axis='y', labelsize=9)

    legend_elements = [
        Line2D([0], [0], color='#d62728', lw=6, label='MD&A text feature'),
        Line2D([0], [0], color='#1f77b4', lw=6, label='Financial / non-financial'),
    ]
    axes[1].legend(handles=legend_elements, loc='lower right', fontsize=9)

    plt.tight_layout()
    save(fig, "fig8_smote_vs_classweight_shap")


# ==================== 主程序 ====================
if __name__ == "__main__":
    print("=" * 60)
    print("Generating 8 figures ...")
    print("Output:", FIG_DIR)
    print("=" * 60)
    fig1_roc()
    fig2_pr()
    fig3_calibration()
    fig4_shap_bar()
    fig5_shap_beeswarm()
    fig6_shap_dependence()
    fig7_shap_stability()
    fig8_smote_vs_cw()
    print("=" * 60)
    print("All done.")

Generating 8 figures ...
Output: D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig1_roc.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig2_pr.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig3_calibration.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig4_shap_bar.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig5_shap_beeswarm.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig6_shap_dependence.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig7_shap_stability.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig8_smote_vs_classweight_shap.png
All done.


In [2]:
# -*- coding: utf-8 -*-
"""
PLOS ONE 财务舞弊识别论文 —— 8 张图完整绘制脚本（特征名已改为英文名称）
环境: Python 3.8.8, matplotlib 3.3.4, seaborn 0.11.1
输出: D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.calibration import calibration_curve

# ==================== 路径 ====================
BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
FIG_DIR = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表"
os.makedirs(FIG_DIR, exist_ok=True)

# ==================== 特征代码 -> 英文名称映射 ====================
FEATURE_NAME_MAP = {
    'F010101A': 'Current Ratio',
    'F010201A': 'Quick Ratio',
    'F010401A': 'Cash Ratio',
    'F010701B': 'Interest Coverage Ratio A',
    'F010801B': 'Operating Cash Flow to Current Liabilities',
    'F011201A': 'Debt to Asset Ratio',
    'F080501A': 'Fixed Assets Growth Rate A',
    'F080601A': 'Total Assets Growth Rate A',
    'F081001B': 'Net Profit Growth Rate A',
    'F081101B': 'Total Profit Growth Rate A',
    'F081201B': 'Operating Profit Growth Rate A',
    'F081601B': 'Operating Revenue Growth Rate A',
    'F082201B': 'Net Operating Cash Flow per Share Growth Rate A',
    'F082701A': 'Equity Growth Rate A',
    'F060101B': 'Net Profit Cash Content',
    'F060301B': 'Operating Revenue Cash Content',
    'F050101B': 'Return on Assets A (ROA A)',
    'F050201B': 'Net Profit to Total Assets (ROA) A',
    'F050301B': 'Net Profit to Current Assets A',
    'F050401B': 'Net Profit to Fixed Assets A',
    'F050501B': 'Return on Equity (ROE) A',
    'F050901B': 'Net Profit to Total Profit',
    'F053201B': 'Long-term Capital Return',
    'F053301B': 'Gross Profit Margin',
    'F051301B': 'Operating Cost Ratio',
    'F051701B': 'Selling Expense Ratio',
    'F053401B': 'R&D Expense Ratio',
    'F052101B': 'Cost and Expense Profit Margin',
    'F053202B': 'Investment Return Ratio',
    'F040101B': 'Accounts Receivable to Revenue',
    'F040201B': 'Accounts Receivable Turnover A',
    'F040401B': 'Inventory to Revenue',
    'F040501B': 'Inventory Turnover A',
    'F040801B': 'Accounts Payable Turnover A',
    'F041201B': 'Current Assets Turnover A',
    'F041401B': 'Fixed Assets Turnover A',
    'F041701B': 'Total Assets Turnover A',
    'F041801B': 'Equity Turnover A',
    'F070101B': 'Financial Leverage',
    'F070201B': 'Operating Leverage',
    'InternationalBig4': 'International Big 4 Auditor',
    'TotalAuditFee': 'Total Audit Fee',
    'ContrshrProportion': 'Controlling Shareholder Proportion',
    'Mngmhldn': 'Management Shareholding Ratio',
    'Boardsize': 'Board Size',
    'IndDirectorRatio': 'Independent Director Ratio',
    'SupervisorSize': 'Supervisory Board Size',
    'Y0301b': 'Change in Share Capital Structure',
    'Y0501b': 'Relatedness among Top 10 Shareholders',
    'ChairmanHoldsharesRatio': 'Chairman Shareholding Ratio',
    'ManagerHoldsharesRatio': 'General Manager Shareholding Ratio',
    'Y1001b': 'Chairman-CEO Duality',
    'LargestHolderRate': 'Largest Holder Rate (%)',
    'TopTenHoldersRate': 'Top Ten Holders Rate (%)',
    'IsDisclosingEvaRep': 'Disclosure of Internal Control Evaluation Report',
    'IsValid': 'Internal Control Effectiveness',
    'IsDeficiency': 'Internal Control Deficiency',
    'TextualSimilarity': 'Textual Similarity',
    'PositiveVocabularyNum': 'Positive Vocabulary Count',
    'NegativeVocabularyNum': 'Negative Vocabulary Count',
    'EmotionTone1': 'Emotion Tone 1',
    'EmotionTone2': 'Emotion Tone 2',
    'PosRatio': 'Positive Word Ratio',
    'NegRatio': 'Negative Word Ratio',
    'SentLenAvg': 'Average Sentence Length',
    'SentLenStd': 'Standard Deviation of Sentence Length',
    'ComplexWordRatio': 'Complex Word Ratio',
    'DigitDensity': 'Digit Density',
    'PuncDensity': 'Punctuation Density',
    'TTR': 'Type-Token Ratio',
    'Jaccard_prev': 'Jaccard Similarity with Previous Year MD&A',
    'EditSim_prev': 'Edit Similarity with Previous Year MD&A',
    'TFIDF_Cosine_prev': 'TF-IDF Cosine Similarity with Previous Year MD&A',
    'DLUT_PosNum': 'DLUT Positive Word Count',
    'DLUT_NegNum': 'DLUT Negative Word Count',
    'DLUT_PosRatio': 'DLUT Positive Word Ratio',
    'DLUT_NegRatio': 'DLUT Negative Word Ratio',
    'DLUT_PosIntensity': 'DLUT Positive Word Intensity',
    'DLUT_NegIntensity': 'DLUT Negative Word Intensity',
    'DLUT_EmotionScore': 'DLUT Emotion Score',
    'DLUT_EmotionTone': 'DLUT Emotion Tone',
    'DLUT_NegAfterNeg': 'DLUT Negative After Negative',
    'DLUT_PosAfterNeg': 'DLUT Positive After Negative',
}

# ==================== 全局样式 ====================
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = False

RNG = np.random.RandomState(42)


def save(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, bbox_inches='tight', dpi=300)
    fig.savefig(pdf, bbox_inches='tight')
    plt.close(fig)
    print("[saved]", png)


# ==================== 图 1: ROC 曲线（不变） ====================
def fig1_roc():
    df = pd.read_csv(os.path.join(BASE, "对比_OOF_predictions.csv"))
    y = df['y_true'].values
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    configs = [
        ('Feature Set A (57 features)',
         ['Base_LightGBM', 'Base_XGBoost', 'Base_RandomForest']),
        ('Feature Set A+B+C (83 features)',
         ['Full_LightGBM', 'Full_XGBoost', 'Full_RandomForest']),
    ]
    colors = {'LightGBM': '#1f77b4', 'XGBoost': '#ff7f0e', 'RandomForest': '#2ca02c'}
    for ax, (title, cols) in zip(axes, configs):
        for col in cols:
            model = col.split('_', 1)[1]
            fpr, tpr, _ = roc_curve(y, df[col].values)
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, label=f'{model} (AUC={roc_auc:.4f})',
                    color=colors[model], lw=1.8)
        ax.plot([0, 1], [0, 1], 'k--', lw=1)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(title, fontsize=11)
        ax.legend(loc='lower right', fontsize=9, frameon=True)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    plt.tight_layout()
    save(fig, "fig1_roc")


# ==================== 图 2: PR 曲线（不变） ====================
def fig2_pr():
    df = pd.read_csv(os.path.join(BASE, "对比_OOF_predictions.csv"))
    y = df['y_true'].values
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    configs = [
        ('Feature Set A (57 features)',
         ['Base_LightGBM', 'Base_XGBoost', 'Base_RandomForest']),
        ('Feature Set A+B+C (83 features)',
         ['Full_LightGBM', 'Full_XGBoost', 'Full_RandomForest']),
    ]
    colors = {'LightGBM': '#1f77b4', 'XGBoost': '#ff7f0e', 'RandomForest': '#2ca02c'}
    baseline = y.mean()
    for ax, (title, cols) in zip(axes, configs):
        for col in cols:
            model = col.split('_', 1)[1]
            prec, rec, _ = precision_recall_curve(y, df[col].values)
            ap = average_precision_score(y, df[col].values)
            ax.plot(rec, prec, label=f'{model} (PR-AUC={ap:.4f})',
                    color=colors[model], lw=1.8)
        ax.axhline(baseline, color='gray', ls='--', lw=1,
                   label=f'Baseline ({baseline:.3f})')
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.set_title(title, fontsize=11)
        ax.legend(loc='upper right', fontsize=9)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    plt.tight_layout()
    save(fig, "fig2_pr")


# ==================== 图 3: 校准曲线（不变） ====================
def fig3_calibration():
    df = pd.read_csv(os.path.join(BASE, "Optuna_calibrated_predictions.csv"))
    y = df['y_true'].values
    raw = df['y_prob_raw'].values
    cal = df['y_prob_calibrated'].values

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    for probs, label, color in [(raw, 'Raw probability', '#d62728'),
                                (cal, 'Isotonic calibrated', '#1f77b4')]:
        pt, pp = calibration_curve(y, probs, n_bins=10, strategy='quantile')
        ax.plot(pp, pt, 'o-', label=label, color=color, lw=1.8, ms=5)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed fraud rate')
    ax.set_title('Calibration curve (Optuna LightGBM, OOF)', fontsize=11)
    ax.legend(loc='upper left', fontsize=9)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    plt.tight_layout()
    save(fig, "fig3_calibration")


# ==================== 图 4: SHAP 全局 bar（名称替换） ====================
def fig4_shap_bar():
    df = pd.read_csv(os.path.join(BASE, "SHAP_global_importance.csv"))
    df['FeatureName'] = df['Feature'].map(FEATURE_NAME_MAP)
    top = df.nsmallest(20, 'Rank').iloc[::-1]
    colors = ['#d62728' if m else '#1f77b4' for m in top['IsMDA']]
    fig, ax = plt.subplots(figsize=(8, 7.5))
    ax.barh(top['FeatureName'], top['MeanAbsSHAP'], color=colors,
            edgecolor='black', linewidth=0.4)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('Top 20 features by global SHAP importance', fontsize=11)
    legend_elements = [
        Line2D([0], [0], color='#d62728', lw=6, label='MD&A text feature'),
        Line2D([0], [0], color='#1f77b4', lw=6, label='Financial / non-financial'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    ax.tick_params(axis='y', labelsize=9)
    plt.tight_layout()
    save(fig, "fig4_shap_bar")


# ==================== 图 5: SHAP beeswarm（名称替换） ====================
def _beeswarm_jitter(x, n_bins=80, rng=None):
    if rng is None:
        rng = np.random.RandomState(42)
    hist, edges = np.histogram(x, bins=n_bins)
    bin_idx = np.digitize(x, edges) - 1
    bin_idx = np.clip(bin_idx, 0, len(hist) - 1)
    density = hist[bin_idx].astype(float)
    density = density / (density.max() + 1e-12)
    jitter = (rng.rand(len(x)) - 0.5) * 0.9 * np.sqrt(density)
    return jitter


def fig5_shap_beeswarm():
    sv = pd.read_csv(os.path.join(BASE, "SHAP_values_all.csv"))
    fv = pd.read_csv(os.path.join(BASE, "SHAP_feature_values.csv"))
    gi = pd.read_csv(os.path.join(BASE, "SHAP_global_importance.csv"))

    assert np.array_equal(sv['y_true'].values, fv['y_true'].values), "y_true 不对齐"

    top15_codes = gi.nsmallest(15, 'Rank')['Feature'].tolist()
    top15_names = [FEATURE_NAME_MAP.get(c, c) for c in top15_codes]

    n = len(sv)
    n_show = min(3000, n)
    idx = RNG.choice(n, n_show, replace=False)

    fig, ax = plt.subplots(figsize=(9, 8))
    cmap = plt.get_cmap('coolwarm')

    for i, (code, name) in enumerate(zip(top15_codes, top15_names)):
        y_pos = len(top15_codes) - 1 - i
        shap_vals = sv[code].values[idx]
        feat_vals = fv[code].values[idx]
        fmin, fmax = np.nanmin(feat_vals), np.nanmax(feat_vals)
        if fmax - fmin < 1e-12:
            c = np.zeros_like(feat_vals)
        else:
            c = (feat_vals - fmin) / (fmax - fmin)
        c = np.clip(c, 0, 1)
        jitter = _beeswarm_jitter(shap_vals, rng=np.random.RandomState(42 + i))
        ax.scatter(shap_vals, y_pos + jitter, c=c, cmap=cmap,
                   s=4, alpha=0.65, edgecolors='none', vmin=0, vmax=1)

    ax.set_yticks(range(len(top15_names)))
    ax.set_yticklabels(top15_names[::-1], fontsize=9)
    ax.axvline(0, color='gray', lw=0.8, ls='--')
    ax.set_xlabel('SHAP value (impact on fraud probability)')
    ax.set_title('SHAP beeswarm - Top 15 features', fontsize=11)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label('Feature value (normalized)', fontsize=9)
    cbar.set_ticks([0, 1]); cbar.set_ticklabels(['Low', 'High'])

    plt.tight_layout()
    save(fig, "fig5_shap_beeswarm")


# ==================== 图 6: SHAP dependence（名称替换） ====================
def fig6_shap_dependence():
    sv = pd.read_csv(os.path.join(BASE, "SHAP_values_all.csv"))
    fv = pd.read_csv(os.path.join(BASE, "SHAP_feature_values.csv"))

    code = 'TextualSimilarity'
    name = FEATURE_NAME_MAP.get(code, code)
    shap_vals = sv[code].values
    feat_vals = fv[code].values

    mask = ~(np.isnan(shap_vals) | np.isnan(feat_vals))
    shap_vals = shap_vals[mask]
    feat_vals = feat_vals[mask]

    n = len(shap_vals)
    n_show = min(8000, n)
    idx = RNG.choice(n, n_show, replace=False)
    x = feat_vals[idx]
    y = shap_vals[idx]

    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    ax.scatter(x, y, s=4, alpha=0.25, color='#1f77b4', edgecolors='none')

    n_bins = 20
    bins = np.quantile(x, np.linspace(0, 1, n_bins + 1))
    bins = np.unique(bins)
    bin_centers, bin_means = [], []
    for j in range(len(bins) - 1):
        m = (x >= bins[j]) & (x < bins[j + 1])
        if m.sum() > 10:
            bin_centers.append(np.median(x[m]))
            bin_means.append(y[m].mean())
    ax.plot(bin_centers, bin_means, 'r-', lw=2, label='Binned mean SHAP')

    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_xlabel(f'{name} (cosine similarity vs. previous year MD&A)')
    ax.set_ylabel(f'SHAP value for {name}')
    ax.set_title(f'SHAP dependence plot: {name}', fontsize=11)
    ax.legend(loc='best', fontsize=9)
    plt.tight_layout()
    save(fig, "fig6_shap_dependence")


# ==================== 图 7: SHAP 5 折稳定性（名称替换） ====================
def fig7_shap_stability():
    df = pd.read_csv(os.path.join(BASE, "SHAP_stability_5fold.csv"))
    df = df.rename(columns={'Unnamed: 0': 'Feature'})
    df['FeatureName'] = df['Feature'].map(FEATURE_NAME_MAP)
    top = df.nsmallest(15, 'rank_mean').iloc[::-1]

    fig, ax = plt.subplots(figsize=(8, 6.5))
    y_pos = np.arange(len(top))
    ax.errorbar(top['mean'], y_pos, xerr=top['std'],
                fmt='o', color='#1f77b4', ecolor='#7f7f7f',
                elinewidth=1.2, capsize=3, ms=5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top['FeatureName'], fontsize=9)
    ax.set_xlabel('Mean |SHAP value| across 5 folds')
    ax.set_title('SHAP importance stability across 5 folds (Top 15)', fontsize=11)
    ax.grid(axis='x', linestyle=':', alpha=0.5)
    plt.tight_layout()
    save(fig, "fig7_shap_stability")


# ==================== 图 8: SMOTE vs class_weight SHAP（名称替换） ====================
def fig8_smote_vs_cw():
    cw = pd.read_csv(os.path.join(BASE, "SHAP_global_importance.csv"))
    sm = pd.read_csv(os.path.join(BASE, "SMOTE_SHAP_global_importance.csv"))

    cw['FeatureName'] = cw['Feature'].map(FEATURE_NAME_MAP)
    sm['FeatureName'] = sm['Feature'].map(FEATURE_NAME_MAP)

    cw_top = cw.nsmallest(15, 'Rank').iloc[::-1]
    sm_top = sm.nsmallest(15, 'Rank').iloc[::-1]

    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))

    axes[0].barh(cw_top['FeatureName'], cw_top['MeanAbsSHAP'],
                 color=['#d62728' if m else '#1f77b4' for m in cw_top['IsMDA']],
                 edgecolor='black', linewidth=0.4)
    axes[0].set_title('class_weight = balanced (main model)', fontsize=11)
    axes[0].set_xlabel('Mean |SHAP value|')
    axes[0].tick_params(axis='y', labelsize=9)

    axes[1].barh(sm_top['FeatureName'], sm_top['MeanAbsSHAP'],
                 color=['#d62728' if m else '#1f77b4' for m in sm_top['IsMDA']],
                 edgecolor='black', linewidth=0.4)
    axes[1].set_title('SMOTE (1:1 oversampling)', fontsize=11)
    axes[1].set_xlabel('Mean |SHAP value|')
    axes[1].tick_params(axis='y', labelsize=9)

    legend_elements = [
        Line2D([0], [0], color='#d62728', lw=6, label='MD&A text feature'),
        Line2D([0], [0], color='#1f77b4', lw=6, label='Financial / non-financial'),
    ]
    axes[1].legend(handles=legend_elements, loc='lower right', fontsize=9)

    plt.tight_layout()
    save(fig, "fig8_smote_vs_classweight_shap")


# ==================== 主程序 ====================
if __name__ == "__main__":
    print("=" * 60)
    print("Generating 8 figures (with English feature names) ...")
    print("Output:", FIG_DIR)
    print("=" * 60)
    fig1_roc()
    fig2_pr()
    fig3_calibration()
    fig4_shap_bar()
    fig5_shap_beeswarm()
    fig6_shap_dependence()
    fig7_shap_stability()
    fig8_smote_vs_cw()
    print("=" * 60)
    print("All done.")

Generating 8 figures (with English feature names) ...
Output: D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig1_roc.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig2_pr.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig3_calibration.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig4_shap_bar.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig5_shap_beeswarm.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig6_shap_dependence.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig7_shap_stability.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig8_smote_vs_classweight_shap.png
All done.
